# Notebook 1 of 4 — Understanding, cleaning and preparing the data

**The question:** on the day a diabetic patient goes home from hospital, how likely are they to come back within 30 days?

In this notebook I get the data ready for the models. I:
1. load the hospital records and look at what is inside,
2. clean them and build a few extra features,
3. split them into training and test parts **by patient**, so the test is fair,
4. save the prepared data for notebooks 2, 3 and 4.

---
## Step 0 — Setup

I fix a random seed (42) at the start. Machine learning uses a lot of random choices, like shuffling a deck of
cards. A fixed seed means the deck is shuffled the same way every time, so anyone who re-runs this notebook gets
exactly my numbers.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# I find the project folder whether Jupyter starts in the project or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

---
## Step 1a — What the data is

The data comes from 130 US hospitals between 1999 and 2008. Every patient has diabetes and stayed at least one night.

The key idea: **one row is one hospital stay, not one person.** It works like supermarket receipts. Every shopping
trip prints a new receipt, and a regular customer collects many receipts with the same loyalty-card number.
Here, every stay is a row (`encounter_id`) and the "loyalty card" is the patient number (`patient_nbr`).

In this file a `?` means "missing", so I tell pandas that when I load it.

In [2]:
df = pd.read_csv(DATA / 'diabetic_data.csv', na_values='?', low_memory=False)

print('Rows, columns     :', df.shape)
print('Different patients:', df['patient_nbr'].nunique())

Rows, columns     : (101766, 50)
Different patients: 71518


There are 101,766 stays but only about 71,500 people, so many patients appear more than once.
This matters a lot in Step 2.

### What I am predicting

The column `readmitted` has three answers: `NO` (did not come back), `>30` (came back after more than 30 days)
and `<30` (came back within 30 days).

In [3]:
class_balance = df['readmitted'].value_counts(normalize=True).round(3)
class_balance

readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64

Only about **11%** of stays end in a readmission within 30 days. That is a rare event, like a smoke alarm going off:
an alarm that *never* rings would be "right" 89% of the time and still be useless. So later I judge the models by how
well they find the rare cases, not by plain accuracy.

### Which columns have gaps

In [4]:
most_missing = df.isna().mean().sort_values(ascending=False).head(8).round(3)
most_missing

weight               0.969
max_glu_serum        0.947
A1Cresult            0.833
medical_specialty    0.491
payer_code           0.396
race                 0.022
diag_3               0.014
diag_2               0.004
dtype: float64

`weight` is almost completely empty (97%), so it cannot help. `medical_specialty` (49%) and `payer_code` (40%) are
often empty too. For the two blood-sugar tests (`A1Cresult`, `max_glu_serum`) "missing" simply means the test was not
done, and *the doctor did not test* is useful information in itself.

### The code book

Some columns store codes instead of words, for example discharge code `1` means "Discharged to home".
`IDs_mapping.csv` is the code book. It holds three small tables one after another, so I read it with a short helper.

In [5]:
def read_id_mapping(path):
    tables, current = {}, None
    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if line in ('', ','):
            current = None
            continue
        code, description = line.split(',', 1)
        if current is None:
            current = code
            tables[current] = {}
        else:
            tables[current][code] = description.strip().strip('"')
    return tables

# The UCI download names this file 'IDS_mapping.csv', so I accept both spellings
mapping_file = DATA / 'IDs_mapping.csv'
if not mapping_file.exists():
    mapping_file = DATA / 'IDS_mapping.csv'
id_mapping = read_id_mapping(mapping_file)

pd.Series(id_mapping['discharge_disposition_id'], name='discharge destination')

1                                    Discharged to home
2     Discharged/transferred to another short term h...
3                         Discharged/transferred to SNF
4                         Discharged/transferred to ICF
5     Discharged/transferred to another type of inpa...
6     Discharged/transferred to home with home healt...
7                                              Left AMA
8     Discharged/transferred to home under care of H...
9             Admitted as an inpatient to this hospital
10    Neonate discharged to another hospital for neo...
11                                              Expired
12    Still patient or expected to return for outpat...
13                                       Hospice / home
14                           Hospice / medical facility
15    Discharged/transferred within this institution...
16    Discharged/transferred/referred another instit...
17    Discharged/transferred/referred to this instit...
18                                              

---
## Step 1b — Removing patients who cannot come back

Some stays ended with the patient dying (codes 11, 19, 20, 21) or going to a hospice (13, 14). These patients can
never be readmitted. If I kept them, the model would learn something absurd: "very sick patient → will not come back".
So I remove them, together with 3 rows where gender is "Unknown/Invalid".

In [6]:
EXPIRED_OR_HOSPICE = [11, 13, 14, 19, 20, 21]

rows_before = len(df)
df = df[~df['discharge_disposition_id'].isin(EXPIRED_OR_HOSPICE)]
df = df[df['gender'] != 'Unknown/Invalid']
rows_removed = rows_before - len(df)

print('Rows removed:', rows_removed)
print('Rows left   :', len(df))

Rows removed: 2426
Rows left   : 99340


---
## Step 1c — The target: a yes/no question

The hospital's decision is yes/no: *follow this patient up, or not.* So I turn the three answers into two:

- `<30` → **1** (came back within 30 days)
- `>30` and `NO` → **0**

I count `>30` as 0 on purpose. A readmission three months later is more likely to be a new problem than a
failure of this discharge.

In [7]:
df['y'] = (df['readmitted'] == '<30').astype(int)
print('Share readmitted within 30 days:', round(df['y'].mean(), 3))

Share readmitted within 30 days: 0.114


The column `readmitted` is the answer itself, so it is never used as an input (Step 1g picks the inputs).

---
## Step 1d — Fixing awkward columns

| Column | What I do | Why |
|---|---|---|
| encounter_id | drop | only a receipt number |
| weight | drop | 97% missing |
| examide, citoglipton | drop | always "No", so they tell the model nothing |
| payer_code | gap → "Unknown" | being unrecorded can itself carry information |
| medical_specialty | gap → "Missing" | same reason |
| race | gap → "Unknown" | I need it for the fairness check |
| admission / discharge / source IDs | turn into text | they are codes, not amounts |
| A1Cresult, max_glu_serum | gap → "None" | "None" means "test not done" |

The ID codes are like football shirt numbers: number 9 is not "more" than number 3. If I left them as numbers, a model
could wrongly treat discharge code 6 as "twice" code 3, so I store them as text.

In [8]:
print(df[['examide', 'citoglipton']].nunique())

df = df.drop(columns=['encounter_id', 'weight', 'examide', 'citoglipton'])

df['payer_code'] = df['payer_code'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Missing')
df['race'] = df['race'].fillna('Unknown')

for col in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    df[col] = df[col].astype(str)

for col in ['A1Cresult', 'max_glu_serum']:
    df[col] = df[col].fillna('None')

examide        1
citoglipton    1
dtype: int64


Both `examide` and `citoglipton` have only one value, which confirms they are safe to drop.

---
## Step 1e — Grouping the diagnosis codes

`diag_1`, `diag_2` and `diag_3` hold medical (ICD-9) codes such as `428` (heart failure) or `250.83` (diabetes).
There are over 700 different codes. That is like sorting a library by exact page number: far too fine to learn from.
I put the codes on 9 sensible shelves instead, following Strack et al. (2014):

| Group | ICD-9 codes |
|---|---|
| Circulatory | 390–459, 785 |
| Respiratory | 460–519, 786 |
| Digestive | 520–579, 787 |
| Diabetes | 250.xx |
| Injury | 800–999 |
| Musculoskeletal | 710–739 |
| Genitourinary | 580–629, 788 |
| Neoplasms | 140–239 |
| Other | everything else, including V and E codes |

In [9]:
def icd_group(code):
    if pd.isna(code):
        return 'Missing'
    code = str(code)
    if code[0] in 'VE':
        return 'Other'
    x = float(code)
    if 390 <= x <= 459 or int(x) == 785: return 'Circulatory'
    if 460 <= x <= 519 or int(x) == 786: return 'Respiratory'
    if 520 <= x <= 579 or int(x) == 787: return 'Digestive'
    if int(x) == 250:                    return 'Diabetes'
    if 800 <= x <= 999:                  return 'Injury'
    if 710 <= x <= 739:                  return 'Musculoskeletal'
    if 580 <= x <= 629 or int(x) == 788: return 'Genitourinary'
    if 140 <= x <= 239:                  return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_grp'] = df[col].apply(icd_group)

df['diag_1_grp'].value_counts()

diag_1_grp
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64

Circulatory problems (heart and blood vessels) are the most common main reason for admission.

---
## Step 1f — A few extra features

I add five columns that a doctor would find sensible:

| New feature | Meaning | Example |
|---|---|---|
| total_prior_visits | outpatient + emergency + inpatient visits in the year before | 0 + 1 + 2 = 3 |
| n_meds_changed | how many diabetes medicines had their dose changed | insulin "Up" → 1 |
| n_meds_used | how many diabetes medicines the patient takes | metformin + insulin → 2 |
| age_num | the age band as one number | `[70-80)` → 75 |
| a1c_tested | 1 if the HbA1c blood test was done | |

The idea behind them: a patient who keeps visiting hospital, or whose medicines keep being changed, is probably
sicker and less stable than one who is not.

In [10]:
MEDS = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
        'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
        'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
        'insulin', 'glyburide-metformin', 'glipizide-metformin',
        'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

df['total_prior_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df['n_meds_changed'] = df[MEDS].isin(['Up', 'Down']).sum(axis=1)
df['n_meds_used'] = (df[MEDS] != 'No').sum(axis=1)
df['age_num'] = df['age'].str.extract(r'\[(\d+)')[0].astype(int) + 5
df['a1c_tested'] = (df['A1Cresult'] != 'None').astype(int)

df[['age', 'age_num', 'total_prior_visits', 'n_meds_changed', 'n_meds_used', 'a1c_tested']].head()

,age,age_num,total_prior_visits,n_meds_changed,n_meds_used,a1c_tested
0,[0-10),5,0,0,0,0
1,[10-20),15,0,1,1,0
2,[20-30),25,3,0,1,0
3,[30-40),35,0,1,1,0
4,[40-50),45,0,0,2,0


---
## Step 1g — Choosing the inputs

I sort the inputs into two kinds, because the models treat them differently:
- **categories** (labels such as gender, discharge destination, diagnosis group),
- **numbers** (amounts such as days in hospital or number of medicines).

`X` holds the inputs, `y` the answer, and `groups` the patient number, which I use only for splitting.
I keep `race` as an input for now so I can measure its effect; in notebook 4 I test removing it.

In [11]:
CAT_COLS = ['race', 'gender', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'payer_code', 'medical_specialty',
            'diag_1_grp', 'diag_2_grp', 'diag_3_grp', 'max_glu_serum', 'A1Cresult',
            'change', 'diabetesMed'] + MEDS

NUM_COLS = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses',
            'total_prior_visits', 'n_meds_changed', 'n_meds_used', 'age_num', 'a1c_tested']

X = df[CAT_COLS + NUM_COLS]
y = df['y']
groups = df['patient_nbr']

print('Inputs:', X.shape[1], f'({len(CAT_COLS)} categories, {len(NUM_COLS)} numbers)')

Inputs: 48 (35 categories, 13 numbers)


---
## Step 2 — A fair split: by patient, not by row

Imagine a teacher who tests students with questions they already saw while revising. The marks would look great,
but they would not show what the students really learned. The same trap exists here. If one stay of a patient is in
training and another stay of the *same* patient is in the test, the model partly "recognises" the person and the test
score looks better than real life. This is called **data leakage**.

So I keep all stays of a patient together, in one part only:

```
All rows (grouped by patient)
 ├── TEST  20%  → locked away and used once, at the very end (notebook 4)
 └── TRAIN 80%  → tuning the models with cross-validation (notebook 2)
       └── split again by patient:
            ├── FIT 85%  → training the final models
            └── VAL 15%  → early stopping, calibration and choosing the threshold
```

Every decision (settings, calibration, threshold) is made without the TEST part, like a sealed exam paper that is
opened only once.

In [12]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
g_tr, g_te = groups.iloc[tr_idx], groups.iloc[te_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
fit_i, val_i = next(gss2.split(X_tr, y_tr, g_tr))

X_fit, X_val = X_tr.iloc[fit_i], X_tr.iloc[val_i]
y_fit, y_val = y_tr.iloc[fit_i], y_tr.iloc[val_i]
g_fit, g_val = g_tr.iloc[fit_i], g_tr.iloc[val_i]

# I check that no patient ends up in two parts
print('Patients in both TRAIN and TEST:', len(set(g_tr) & set(g_te)))
print('Patients in both FIT and VAL   :', len(set(g_fit) & set(g_val)))
assert len(set(g_tr) & set(g_te)) == 0 and len(set(g_fit) & set(g_val)) == 0

Patients in both TRAIN and TEST: 0
Patients in both FIT and VAL   : 0


In [13]:
split_sizes = pd.DataFrame({
    'part': ['TRAIN (all)', 'FIT', 'VAL', 'TEST'],
    'rows': [len(y_tr), len(y_fit), len(y_val), len(y_te)],
    'patients': [g_tr.nunique(), g_fit.nunique(), g_val.nunique(), g_te.nunique()],
    'share readmitted <30': [y_tr.mean(), y_fit.mean(), y_val.mean(), y_te.mean()],
}).round(3)
split_sizes

,part,rows,patients,share readmitted <30
0,TRAIN (all),79567,55989,0.114
1,FIT,67565,47590,0.114
2,VAL,12002,8399,0.112
3,TEST,19773,13998,0.113


No patient is shared between parts, and every part keeps about 11% readmissions, so each part is a fair miniature of
the whole data.

### Keeping the patient groups for the fairness check

For every TEST row I keep race, gender and an age band in a separate table. In notebook 4 I use it to check whether
the model works equally well for each group.

In [14]:
sens_te = df.iloc[te_idx][['race', 'gender']].copy()
sens_te['age_band'] = pd.cut(df.iloc[te_idx]['age_num'], [0, 40, 60, 80, 100],
                             labels=['<40', '40-60', '60-80', '80+'])

sens_te['age_band'].value_counts().sort_index()

age_band
<40      1289
40-60    5376
60-80    9277
80+      3831
Name: count, dtype: int64

---
## Saving the prepared data

I save all the parts in one file, `results/prepared_data.joblib`. Notebooks 2, 3 and 4 start by loading it.

In [15]:
prepared = {
    'X_tr': X_tr, 'y_tr': y_tr, 'g_tr': g_tr,
    'X_fit': X_fit, 'y_fit': y_fit,
    'X_val': X_val, 'y_val': y_val,
    'X_te': X_te, 'y_te': y_te,
    'sens_te': sens_te,
    'cat_cols': CAT_COLS, 'num_cols': NUM_COLS,
    'id_mapping': id_mapping,
    'rows_removed': rows_removed,
}
joblib.dump(prepared, RESULTS / 'prepared_data.joblib')
print('Saved results/prepared_data.joblib')

Saved results/prepared_data.joblib
